# Arm B — Clinically Defined Subgroups (Sex-Stratified)

This notebook implements Arm B of the study: patients are partitioned into two clinically defined subgroups by sex, and a classifier is trained independently within each subgroup. Arm B addresses RQ1 by comparing sex-stratified modelling against the non-stratified baseline established in Arm A (`global_model_armA.ipynb`).

Sex is used as the stratification variable because cardiovascular disease can differ between males and females in risk-factor profiles and feature–outcome relationships, with sex-specific models reported in the literature to differ in predictive performance and important predictors (proposal, Section 3.3). As a binary variable, sex also produces exactly two subgroups, which helps keep each subgroup large enough for reliable training given the limited sample size.

Arm B reuses Arm A's dataset, feature definitions, preprocessing, classifiers, hyperparameter grids, nested cross-validation procedure, evaluation metrics, and outer fold partitions unchanged. The only methodological difference from Arm A is that a separate logistic regression and random forest are fitted for the male subgroup and for the female subgroup, instead of one model fitted on the full population. Predictions from both subgroup models are pooled back into a single population-level validation set for the primary comparison against Arm A, as required by the proposal (Section 3.5).

## 2. Imports and configuration

`RANDOM_STATE` is fixed for reproducibility, and `K_OUTER = K_INNER = 5`, for the same reasons documented in Arm A (a small dataset, balancing per-fold sample size against stability). These values match Arm A exactly so that the inner cross-validation procedure is identical across arms.

In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Configuration (must match Arm A)
RANDOM_STATE = 42
K_OUTER      = 5
K_INNER      = 5

## 3. Load dataset

Arm B uses the same cleaned Cleveland extract as Arm A. The cleaning procedure (missing-value handling, duplicate check, and the resulting 303 → 297 record count) is documented in `data_cleaning.ipynb`; it is not repeated here, since Arm B must operate on the identical dataset rather than an independently re-derived one.

In [2]:
df = pd.read_csv("heart+disease/cleveland_clean.csv")
print("Loaded shape:", df.shape)
assert df.shape[0] == 297, "Arm B expects the same 297-record cleaned dataset used in Arm A."
df.head()

Loaded shape: (297, 15)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,check
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,False
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,True
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,True
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,False
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,False


## 4. Target construction

The original Cleveland target, `num`, represents the presence/severity of heart disease. For binary classification, observations with `num = 0` are coded as class 0 (absence of disease), while observations with `num > 0` are coded as class 1 (presence of disease). The derived `check` column is excluded from the feature matrix together with `num`, for the same leakage reason documented in Arm A.

In [3]:
target_col = "num"

y = (df[target_col] > 0).astype(int)
X = df.drop(columns=[target_col, "check"], errors="ignore")

print("Samples:", len(df), " Features:", X.shape[1])
print("Class balance:")
print(y.value_counts().rename({0: "no disease", 1: "disease"}))
print("Positive rate: {:.3f}".format(y.mean()))

Samples: 297  Features: 13
Class balance:
num
no disease    160
disease       137
Name: count, dtype: int64
Positive rate: 0.461


## 5. Feature definition

Identical to Arm A: continuous features are standardised, nominal category codes are one-hot encoded, and binary/count features pass through unchanged. `sex` is retained as a passthrough feature for consistency with Arm A's feature definition, even though `sex` is constant within each sex-specific subgroup once the data are split and therefore cannot carry any discriminatory information inside a subgroup. It is kept rather than dropped because the proposal requires an identical feature set across arms, and a constant passthrough column has no effect on `StandardScaler` or `OneHotEncoder` behaviour for the other features.

In [4]:
continuous  = ["age", "trestbps", "chol", "thalach", "oldpeak"]
nominal     = ["cp", "restecg", "slope", "thal"]
passthrough = ["sex", "fbs", "exang", "ca"]

print("Continuous:", continuous)
print("Nominal (one-hot encoded):", nominal)
print("Passthrough (already binary/count):", passthrough)

Continuous: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Nominal (one-hot encoded): ['cp', 'restecg', 'slope', 'thal']
Passthrough (already binary/count): ['sex', 'fbs', 'exang', 'ca']


## 6. Sex-stratified exploratory summary

Arm B partitions patients into exactly two subgroups using `sex`: `sex = 1` (male) and `sex = 0` (female). No additional clinical subgroups are created. The summary below reports subgroup sizes and disease rates, confirming that both subgroups are large and contain both outcome classes, which is required for stratified cross-validation to be well-defined within each subgroup. Patients are not removed to equalise subgroup sizes.

In [5]:
sex_label = df["sex"].map({1.0: "male", 0.0: "female"})

summary = pd.DataFrame({
    "n": sex_label.value_counts(),
    "no_disease": df.loc[y == 0, "sex"].map({1.0: "male", 0.0: "female"}).value_counts(),
    "disease": df.loc[y == 1, "sex"].map({1.0: "male", 0.0: "female"}).value_counts(),
}).fillna(0).astype(int)
summary["disease_rate"] = (summary["disease"] / summary["n"]).round(3)
summary.loc["total"] = [len(df), int((y == 0).sum()), int((y == 1).sum()), round(y.mean(), 3)]
summary

,n,no_disease,disease,disease_rate
sex,,,,
male,201.0,89.0,112.0,0.557
female,96.0,71.0,25.0,0.260
total,297.0,160.0,137.0,0.461


## 7. Preprocessing

Preprocessing that estimates parameters from the data is kept inside a scikit-learn `Pipeline` together with each classifier, exactly as in Arm A. Because a separate pipeline is fitted per outer fold per sex subgroup (Section 10), the scaler and encoder only ever see the training portion of that subgroup, preventing leakage from validation data. The `ColumnTransformer` definition is identical to Arm A's.

In [6]:
# Fitted only inside the pipeline, on training-fold data (see Section 10), never on the full dataset.
preprocess = ColumnTransformer([
    ("num",  StandardScaler(),                       continuous),
    ("cat",  OneHotEncoder(handle_unknown="ignore"),  nominal),
    ("pass", "passthrough",                           passthrough),
])

## 8. Reuse Arm A's outer folds

The proposal requires identical outer cross-validation fold partitions across all arms (Section 3.5). Arm A already created and saved this partition to `fold_id.csv`; Arm B loads it directly rather than generating a new one, so that every patient sits in exactly the same outer fold in Arm A and Arm B. Sex-specific train/validation masks are then derived by combining the shared fold assignment with the `sex` variable. The check below confirms that every fold, for both sexes, contains enough patients and both outcome classes for nested cross-validation to be well-defined.

In [7]:
FOLD_FILE = "fold_id.csv"
if not os.path.exists(FOLD_FILE):
    raise FileNotFoundError(
        f"{FOLD_FILE} not found. Arm B requires the outer fold partition created by "
        "global_model_armA.ipynb; run that notebook first."
    )

fold_id = pd.read_csv(FOLD_FILE)["fold"].to_numpy()
if len(fold_id) != len(df):
    raise ValueError(
        f"{FOLD_FILE} has {len(fold_id)} entries but the current dataset has {len(df)} rows."
    )
print(f"Loaded outer fold assignment from {FOLD_FILE} (shared with Arm A).")

sex = df["sex"].to_numpy()  # 1 = male, 0 = female

for k in range(K_OUTER):
    train, validation = (fold_id != k), (fold_id == k)
    for group_name, code in {"male": 1, "female": 0}.items():
        y_train_g = y[train & (sex == code)]
        y_val_g = y[validation & (sex == code)]
        print(f"fold {k} {group_name:6s} | train n={len(y_train_g):3d} classes={y_train_g.value_counts().to_dict()}"
              f" | validation n={len(y_val_g):3d} classes={y_val_g.value_counts().to_dict()}")

Loaded outer fold assignment from fold_id.csv (shared with Arm A).
fold 0 male   | train n=157 classes={1: 90, 0: 67} | validation n= 44 classes={1: 22, 0: 22}
fold 0 female | train n= 80 classes={0: 61, 1: 19} | validation n= 16 classes={0: 10, 1: 6}
fold 1 male   | train n=160 classes={1: 89, 0: 71} | validation n= 41 classes={1: 23, 0: 18}
fold 1 female | train n= 77 classes={0: 57, 1: 20} | validation n= 19 classes={0: 14, 1: 5}
fold 2 male   | train n=162 classes={1: 89, 0: 73} | validation n= 39 classes={1: 23, 0: 16}
fold 2 female | train n= 76 classes={0: 55, 1: 21} | validation n= 20 classes={0: 16, 1: 4}
fold 3 male   | train n=159 classes={1: 90, 0: 69} | validation n= 42 classes={1: 22, 0: 20}
fold 3 female | train n= 79 classes={0: 59, 1: 20} | validation n= 17 classes={0: 12, 1: 5}
fold 4 male   | train n=166 classes={1: 90, 0: 76} | validation n= 35 classes={1: 22, 0: 13}
fold 4 female | train n= 72 classes={0: 52, 1: 20} | validation n= 24 classes={0: 19, 1: 5}


## 9. Model definitions and hyperparameter grids

Exactly the same two classifiers and hyperparameter grids as Arm A. Male and female models are tuned from the same candidate hyperparameters and the same selection procedure as the global model; only the data used to fit them differs.

In [8]:
models = {
    "logreg": (
        Pipeline([("pre", preprocess),
                  ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "rf": (
        Pipeline([("pre", preprocess),
                  ("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 5, 10]},
    ),
}

## 10. Nested cross-validation within sex subgroups

For each model and each outer fold, a separate model is trained for the male subgroup and for the female subgroup. Within each subgroup, hyperparameters are selected using `GridSearchCV` with an inner stratified cross-validation on that subgroup's outer-training patients only; the outer validation fold is never used for hyperparameter selection, exactly as in Arm A. ROC-AUC is used as the tuning criterion (`scoring="roc_auc"`), matching Arm A, because it is threshold-independent and is one of the three metrics reported for all arms. The selected model is refit on the full outer-training portion of that subgroup and used to predict the corresponding validation patients. Predicted class labels use the same fixed probability threshold of 0.5 as Arm A, applied identically to both sexes and both models, so that thresholding is not a source of variation between arms.

Male and female models are fitted completely independently: each subgroup's inner cross-validation only ever sees that subgroup's data, so a male model can never be tuned or validated using female patients, and vice versa. Because the two subgroups differ in composition, the selected hyperparameters may differ between sexes; this is expected and is not itself evidence of a methodological inconsistency, since both subgroups draw from the same grids and selection metric.

For every outer fold, the male model's predictions on the male validation patients and the female model's predictions on the female validation patients are pooled into a single population-level validation set before computing accuracy, F1, and ROC-AUC. This pooled, fold-level metric — not an average of the two subgroups' separate scores — is the primary Arm B result, since it is what is directly comparable to Arm A's population-level evaluation.

In [9]:
fold_results = []       # pooled, population-level: primary result
sex_fold_results = []   # sex-specific: secondary, descriptive result
oof_rows = []
patient_id = df.index.to_numpy()
sex_groups = {"male": 1, "female": 0}

for name, (pipe, grid) in models.items():
    for k in range(K_OUTER):
        train, validation = (fold_id != k), (fold_id == k)

        pooled_y_true, pooled_proba, pooled_pred = [], [], []

        for group_name, sex_code in sex_groups.items():
            group_train = train & (sex == sex_code)
            group_validation = validation & (sex == sex_code)

            # Hyperparameter selection uses only this subgroup's outer-training
            # patients; the outer validation fold is not seen until scoring below.
            inner = StratifiedKFold(n_splits=K_INNER, shuffle=True, random_state=RANDOM_STATE)
            search = GridSearchCV(pipe, grid, cv=inner, scoring="roc_auc", n_jobs=-1)
            search.fit(X[group_train], y[group_train])

            best = search.best_estimator_
            proba = best.predict_proba(X[group_validation])[:, 1]
            # Predicted probability of the positive class (heart disease present).
            pred = (proba >= 0.5).astype(int)  # Fixed threshold, consistent with Arm A.

            y_val = y[group_validation].to_numpy()

            sex_fold_results.append({
                "model": name, "fold": k, "sex": group_name,
                "n": int(group_validation.sum()),
                "accuracy": accuracy_score(y_val, pred),
                "f1": f1_score(y_val, pred),
                "roc_auc": roc_auc_score(y_val, proba) if len(np.unique(y_val)) > 1 else np.nan,
                "best_params": search.best_params_,
            })

            for pid, yt, p, c in zip(patient_id[group_validation], y_val, proba, pred):
                oof_rows.append({
                    "patient_id": int(pid), "fold": int(k), "sex": group_name,
                    "y_true": int(yt), "model": name, "proba": float(p), "pred": int(c),
                })

            pooled_y_true.append(y_val)
            pooled_proba.append(proba)
            pooled_pred.append(pred)

        # Population-level pooled result for this outer fold: male and female
        # validation predictions are combined before scoring, not averaged,
        # so the pooled metric reflects the full validation fold at once.
        y_true_pooled = np.concatenate(pooled_y_true)
        proba_pooled = np.concatenate(pooled_proba)
        pred_pooled = np.concatenate(pooled_pred)

        fold_results.append({
            "model": name,
            "fold": k,
            "accuracy": accuracy_score(y_true_pooled, pred_pooled),
            "f1": f1_score(y_true_pooled, pred_pooled),
            "roc_auc": roc_auc_score(y_true_pooled, proba_pooled),
        })

fold_results_df = pd.DataFrame(fold_results)
sex_fold_results_df = pd.DataFrame(sex_fold_results)
print("Pooled nested cross-validation complete:", len(fold_results_df), "model x fold rows")
print("Sex-specific nested cross-validation complete:", len(sex_fold_results_df), "model x fold x sex rows")

Pooled nested cross-validation complete: 10 model x fold rows
Sex-specific nested cross-validation complete: 20 model x fold x sex rows


## 11. Fold-level results

The pooled, population-level fold results are the primary output of this notebook and are saved for comparison with Arm A. Sex-specific fold-level results are also shown as a secondary, descriptive breakdown, but are not the basis for the Arm A vs Arm B comparison.

In [10]:
fold_results_df.to_csv("armB_fold_results.csv", index=False)
print("Saved armB_fold_results.csv (pooled, primary)")
display_pooled = fold_results_df.round(3)
display_pooled

Saved armB_fold_results.csv (pooled, primary)


,model,fold,accuracy,f1,roc_auc
0,logreg,0,0.817,0.792,0.901
1,logreg,1,0.783,0.764,0.875
2,logreg,2,0.746,0.727,0.866
3,logreg,3,0.797,0.760,0.899
4,logreg,4,0.780,0.723,0.905
5,rf,0,0.867,0.862,0.945
6,rf,1,0.783,0.772,0.900
7,rf,2,0.678,0.655,0.843
8,rf,3,0.780,0.735,0.887
9,rf,4,0.864,0.840,0.917


In [11]:
sex_fold_results_display = sex_fold_results_df.drop(columns=["best_params"]).round(3)
sex_fold_results_display

,model,fold,sex,n,accuracy,f1,roc_auc
0,logreg,0,male,44,0.886,0.894,0.961
1,logreg,0,female,16,0.625,0.000,0.933
2,logreg,1,male,41,0.756,0.783,0.855
3,logreg,1,female,19,0.842,0.667,0.929
4,logreg,2,male,39,0.667,0.723,0.793
5,logreg,2,female,20,0.900,0.750,0.859
6,logreg,3,male,42,0.738,0.732,0.855
7,logreg,3,female,17,0.941,0.889,0.983
8,logreg,4,male,35,0.771,0.810,0.906
9,logreg,4,female,24,0.792,0.000,0.800


## 12. Overall performance summary

Mean and standard deviation across the 5 outer folds are reported for each model, using the pooled population-level fold results, so that Arm B's primary comparison with Arm A rests on the same kind of summary statistic in both notebooks. The corresponding sex-specific summary (Male LR, Male RF, Female LR, Female RF) is reported separately below as a secondary, descriptive result.

In [12]:
results = (
    fold_results_df
    .groupby("model")[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
results = results.reset_index()

for _, row in results.iterrows():
    print(f"{row['model']:7s} | ACC {row['accuracy_mean']:.3f} +/- {row['accuracy_std']:.3f}"
          f" | F1 {row['f1_mean']:.3f} +/- {row['f1_std']:.3f}"
          f" | AUC {row['rocauc_mean']:.3f} +/- {row['rocauc_std']:.3f}")

results.to_csv("armB_results.csv", index=False)
print("Saved armB_results.csv (pooled, primary)")
results.set_index("model").round(3)

logreg  | ACC 0.784 +/- 0.026 | F1 0.753 +/- 0.029 | AUC 0.889 +/- 0.018
rf      | ACC 0.794 +/- 0.077 | F1 0.773 +/- 0.084 | AUC 0.898 +/- 0.038
Saved armB_results.csv (pooled, primary)


,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
model,,,,,,
logreg,0.784,0.026,0.753,0.029,0.889,0.018
rf,0.794,0.077,0.773,0.084,0.898,0.038


In [13]:
sex_results = (
    sex_fold_results_df
    .groupby(["sex", "model"])[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
sex_results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
sex_results = sex_results.reset_index()

for _, row in sex_results.iterrows():
    label = f"{row['sex']} {row['model']}"
    print(f"{label:13s} | ACC {row['accuracy_mean']:.3f} +/- {row['accuracy_std']:.3f}"
          f" | F1 {row['f1_mean']:.3f} +/- {row['f1_std']:.3f}"
          f" | AUC {row['rocauc_mean']:.3f} +/- {row['rocauc_std']:.3f}")

sex_results.to_csv("armB_sex_specific_results.csv", index=False)
print("Saved armB_sex_specific_results.csv (secondary, descriptive)")
sex_results.round(3)

female logreg | ACC 0.820 +/- 0.123 | F1 0.461 +/- 0.428 | AUC 0.901 +/- 0.072
female rf     | ACC 0.875 +/- 0.021 | F1 0.727 +/- 0.058 | AUC 0.930 +/- 0.079
male logreg   | ACC 0.764 +/- 0.079 | F1 0.788 +/- 0.069 | AUC 0.874 +/- 0.063
male rf       | ACC 0.756 +/- 0.121 | F1 0.783 +/- 0.101 | AUC 0.863 +/- 0.099
Saved armB_sex_specific_results.csv (secondary, descriptive)


,sex,model,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
0,female,logreg,0.820,0.123,0.461,0.428,0.901,0.072
1,female,rf,0.875,0.021,0.727,0.058,0.930,0.079
2,male,logreg,0.764,0.079,0.788,0.069,0.874,0.063
3,male,rf,0.756,0.121,0.783,0.101,0.863,0.099


## 13. Out-of-fold predictions

For every patient, the out-of-fold prediction from the one outer fold in which they were held out is retained, together with the patient identifier, fold id, sex, true label, model name, predicted probability, and predicted class. This matches Arm A's out-of-fold prediction structure (`patient_id`, `fold`, `y_true`, `model`, `proba`, `pred`) with `sex` added, so that Arm A, Arm B, and Arm C can be compared using the same patient identifiers, fold assignment, and column layout. Because each patient belongs to exactly one sex, they receive exactly one out-of-fold prediction per model, from the subgroup model that held them out.

In [14]:
oof_df = pd.DataFrame(oof_rows)

# Verify exactly one OOF prediction per patient per model, and full coverage of all patients.
counts_per_model = oof_df.groupby("model")["patient_id"].nunique()
assert (counts_per_model == len(df)).all(), "Every patient must receive exactly one OOF prediction per model."
assert oof_df.groupby(["model", "patient_id"]).size().max() == 1, "Duplicate OOF prediction detected for a patient."

oof_df.to_csv("armB_predictions.csv", index=False)
print("Saved armB_predictions.csv:", oof_df.shape)
oof_df.head()

Saved armB_predictions.csv: (594, 7)


,patient_id,fold,sex,y_true,model,proba,pred
0,1,0,male,1,logreg,0.968202,1
1,2,0,male,1,logreg,0.958201,1
2,8,0,male,1,logreg,0.856343,1
3,10,0,male,0,logreg,0.481033,0
4,12,0,male,1,logreg,0.623056,1


## 14. Interpretation / notes

The primary comparison for RQ1 is Arm A's global model (`armA_results.csv`) against Arm B's pooled, sex-stratified model (`armB_results.csv`); both report accuracy, F1-score, and ROC-AUC as mean ± standard deviation over the same 5 outer folds. The sex-specific results (`armB_sex_specific_results.csv`) are retained only as a secondary, descriptive breakdown of where any difference between Arm A and Arm B originates; they are not themselves the basis for answering RQ1. Whether Arm B outperforms, underperforms, or performs similarly to Arm A is an empirical outcome of this comparison and is not assumed in advance.

For comparability with Arm C: the same `fold_id.csv` partition, the same feature grouping, and the same classifiers/grids used here should be reused unchanged, so that Arm C's data-driven clusters differ from Arm A and Arm B only in how patients are grouped, not in any other modelling condition.